## Dataset

Loads the enriched train/test CSVs saved at the end of notebook 04 (`enriched_train.csv` / `enriched_test.csv`), which already reflect the **one-month-forward split**: train = `CloseDate < 2026-06-01`, test = `CloseDate >= 2026-06-01`.

In [1]:
# Week 8 - Evaluation Expansion
#
# Goal: go beyond R2 - report MAPE and MdAPE for every model tried in Weeks 5-7, and check
# whether performance is stable across price bands rather than just looking at one top-line number.
#
# Every model below reuses the hyperparameters already selected in notebooks 04/05 (no new tuning
# happens here) so these numbers are directly comparable to the Week 5-7 results.

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

train = pd.read_csv('/Users/jgd/IDXWORK/datasets/enriched/enriched_train.csv', low_memory=False)
test = pd.read_csv('/Users/jgd/IDXWORK/datasets/enriched/enriched_test.csv', low_memory=False)

print(train.shape)
print(test.shape)

(125795, 86)
(12408, 86)


In [3]:
# Same final feature set as the end of Week 6/7
numerical_features = ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet',
                      'YearBuilt', 'Latitude', 'Longitude', 'AssociationFee', 'Stories',
                      'BedBathRatio', 'PropertyAge', 'BedroomAreaRatio', 'CloseMonthSin', 'CloseMonthCos']

boolean_features = ['FireplaceYN', 'NewConstructionYN', 'AttachedGarageYN', 'ViewYN', 'PoolPrivateYN']

categorical_features = ['CountyOrParish', 'DistrictGrouped']

target = 'ClosePrice'

features = numerical_features + boolean_features + categorical_features

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

In [4]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_features + boolean_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(X_train_processed.shape)
print(X_test_processed.shape)

(125795, 175)
(12408, 175)


## Refitting Every Model From Weeks 5-7

Same final hyperparameters as before - nothing is re-tuned here. This just gets every model's predictions in one place so metrics (including the price-band breakdown below) can be computed consistently on the same split.

In [5]:
from sklearn.linear_model import LinearRegression

model_lr = LinearRegression()
model_lr.fit(X_train_processed, y_train)
y_pred_lr = model_lr.predict(X_test_processed)

r2_lr = r2_score(y_test, y_pred_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
mape_lr = mean_absolute_percentage_error(y_test, y_pred_lr)
mdape_lr = np.median(np.abs((y_test - y_pred_lr) / y_test))

print(f"Linear Regression R2: {r2_lr:.4f}")
print(f"MAE: ${mae_lr:,.0f}")
print(f"MAPE: {mape_lr:.2%}")
print(f"MdAPE: {mdape_lr:.2%}")

Linear Regression R2: 0.6901
MAE: $332,998
MAPE: 30.35%
MdAPE: 21.98%


In [6]:
y_train_log = np.log(y_train)

model_log = LinearRegression()
model_log.fit(X_train_processed, y_train_log)
y_pred_log = np.exp(model_log.predict(X_test_processed))

r2_log = r2_score(y_test, y_pred_log)
mae_log = mean_absolute_error(y_test, y_pred_log)
mape_log = mean_absolute_percentage_error(y_test, y_pred_log)
mdape_log = np.median(np.abs((y_test - y_pred_log) / y_test))

# R2 computed on the log scale too - this is the number notebook 04 reports for this model.
# It looks much stronger (~0.81) because it's scoring log(price) predictions against log(price),
# not price against price. Both are printed here so the two notebooks' numbers don't look
# inconsistent - they're the same model, just two different R2 bases.

r2_log_logscale = r2_score(np.log(y_test), model_log.predict(X_test_processed))

print(f"Log-transformed Linear Regression R2 (dollar scale): {r2_log:.4f}")
print(f"Log-transformed Linear Regression R2 (log scale, matches notebook 04): {r2_log_logscale:.4f}")
print(f"MAE: ${mae_log:,.0f}")
print(f"MAPE: {mape_log:.2%}")
print(f"MdAPE: {mdape_log:.2%}")

Log-transformed Linear Regression R2 (dollar scale): 0.6114
Log-transformed Linear Regression R2 (log scale, matches notebook 04): 0.8147
MAE: $285,860
MAPE: 20.29%
MdAPE: 14.65%


In [7]:
from sklearn.tree import DecisionTreeRegressor

model_dt = DecisionTreeRegressor(max_depth=15, random_state=42)
model_dt.fit(X_train_processed, y_train)
y_pred_dt = model_dt.predict(X_test_processed)

r2_dt = r2_score(y_test, y_pred_dt)
mae_dt = mean_absolute_error(y_test, y_pred_dt)
mape_dt = mean_absolute_percentage_error(y_test, y_pred_dt)
mdape_dt = np.median(np.abs((y_test - y_pred_dt) / y_test))

print(f"Decision Tree (depth 15) R2: {r2_dt:.4f}")
print(f"MAE: ${mae_dt:,.0f}")
print(f"MAPE: {mape_dt:.2%}")
print(f"MdAPE: {mdape_dt:.2%}")

Decision Tree (depth 15) R2: 0.7577
MAE: $239,687
MAPE: 18.14%
MdAPE: 11.83%


In [8]:
from sklearn.ensemble import RandomForestRegressor

model_rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42)
model_rf.fit(X_train_processed, y_train)
y_pred_rf = model_rf.predict(X_test_processed)

r2_rf = r2_score(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mape_rf = mean_absolute_percentage_error(y_test, y_pred_rf)
mdape_rf = np.median(np.abs((y_test - y_pred_rf) / y_test))

print(f"Random Forest (100 trees, depth 15) R2: {r2_rf:.4f}")
print(f"MAE: ${mae_rf:,.0f}")
print(f"MAPE: {mape_rf:.2%}")
print(f"MdAPE: {mdape_rf:.2%}")

Random Forest (100 trees, depth 15) R2: 0.8568
MAE: $197,695
MAPE: 15.76%
MdAPE: 10.23%


In [9]:
from xgboost import XGBRegressor

model_xgb = XGBRegressor(n_estimators=200, max_depth=10, learning_rate=0.1, random_state=42)
model_xgb.fit(X_train_processed, y_train)
y_pred_xgb = model_xgb.predict(X_test_processed)

r2_xgb = r2_score(y_test, y_pred_xgb)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
mape_xgb = mean_absolute_percentage_error(y_test, y_pred_xgb)
mdape_xgb = np.median(np.abs((y_test - y_pred_xgb) / y_test))

print(f"XGBoost (tuned) R2: {r2_xgb:.4f}")
print(f"MAE: ${mae_xgb:,.0f}")
print(f"MAPE: {mape_xgb:.2%}")
print(f"MdAPE: {mdape_xgb:.2%}")

XGBoost (tuned) R2: 0.8851
MAE: $167,793
MAPE: 12.70%
MdAPE: 8.48%


In [10]:
from lightgbm import LGBMRegressor

model_lgbm = LGBMRegressor(n_estimators=200, max_depth=15, learning_rate=0.2, random_state=42, verbose=-1)
model_lgbm.fit(X_train_processed, y_train)
y_pred_lgbm = model_lgbm.predict(X_test_processed)

r2_lgbm = r2_score(y_test, y_pred_lgbm)
mae_lgbm = mean_absolute_error(y_test, y_pred_lgbm)
mape_lgbm = mean_absolute_percentage_error(y_test, y_pred_lgbm)
mdape_lgbm = np.median(np.abs((y_test - y_pred_lgbm) / y_test))

print(f"LightGBM (tuned) R2: {r2_lgbm:.4f}")
print(f"MAE: ${mae_lgbm:,.0f}")
print(f"MAPE: {mape_lgbm:.2%}")
print(f"MdAPE: {mdape_lgbm:.2%}")

LightGBM (tuned) R2: 0.8868
MAE: $180,152
MAPE: 14.49%
MdAPE: 10.32%


## Overall Metrics Summary

R2, MAE, MAPE, and MdAPE for every model, side by side. Exported to `metrics_summary.csv` per the Week 8 deliverable.

**Note on the log-transformed model:** R2 here is computed on the back-transformed (dollar-scale) predictions rather than the log scale used internally, so every row of this table sits on the same basis for a fair comparison - this number won't exactly match the log-scale R2 reported in notebook 04.

In [11]:
metrics_summary = pd.DataFrame({
    'Model': ['Linear Regression', 'Log-transformed Linear Regression', 'Decision Tree (depth 15)',
              'Random Forest (100 trees, depth 15)', 'XGBoost (tuned)', 'LightGBM (tuned)'],
    'R2': [r2_lr, r2_log, r2_dt, r2_rf, r2_xgb, r2_lgbm],
    'R2 (log scale)': [None, r2_log_logscale, None, None, None, None],
    'MAE': [mae_lr, mae_log, mae_dt, mae_rf, mae_xgb, mae_lgbm],
    'MAPE': [mape_lr, mape_log, mape_dt, mape_rf, mape_xgb, mape_lgbm],
    'MdAPE': [mdape_lr, mdape_log, mdape_dt, mdape_rf, mdape_xgb, mdape_lgbm],
})

metrics_summary.to_csv('metrics_summary.csv', index=False)

display_summary = metrics_summary.copy()
display_summary['R2'] = display_summary['R2'].map(lambda x: f"{x:.4f}")
display_summary['R2 (log scale)'] = display_summary['R2 (log scale)'].map(lambda x: f"{x:.4f}" if pd.notnull(x) else '-')
display_summary['MAE'] = display_summary['MAE'].map(lambda x: f"${x:,.0f}")
display_summary['MAPE'] = display_summary['MAPE'].map(lambda x: f"{x:.2%}")
display_summary['MdAPE'] = display_summary['MdAPE'].map(lambda x: f"{x:.2%}")

display_summary

,Model,R2,R2 (log scale),MAE,MAPE,MdAPE
0,Linear Regression,0.6901,-,"$332,998",30.35%,21.98%
1,Log-transformed Linear Regression,0.6114,0.8147,"$285,860",20.29%,14.65%
2,Decision Tree (depth 15),0.7577,-,"$239,687",18.14%,11.83%
3,"Random Forest (100 trees, depth 15)",0.8568,-,"$197,695",15.76%,10.23%
4,XGBoost (tuned),0.8851,-,"$167,793",12.70%,8.48%
5,LightGBM (tuned),0.8868,-,"$180,152",14.49%,10.32%


## Price-Band Breakdown

Splitting the test set into 5 equal-count price quintiles, computed directly on the test set's own `ClosePrice` (a post-hoc reporting split, not a preprocessing step - it doesn't touch training and introduces no leakage), and reporting MAPE/MdAPE per model within each band. This checks whether error is systematically worse for luxury or entry-level homes, per the best-practices doc's recommendation.

In [12]:
price_bands = pd.qcut(y_test, q=5, labels=['Q1 (lowest)', 'Q2', 'Q3', 'Q4', 'Q5 (highest)'], duplicates='drop')

band_edges = pd.qcut(y_test, q=5, duplicates='drop').cat.categories
for label, interval in zip(price_bands.cat.categories, band_edges):
    print(f"{label}: ${interval.left:,.0f} - ${interval.right:,.0f}")

Q1 (lowest): $185,000 - $575,000
Q2: $575,000 - $800,000
Q3: $800,000 - $1,100,000
Q4: $1,100,000 - $1,650,000
Q5 (highest): $1,650,000 - $8,920,000


In [13]:
def band_metrics(y_true, y_pred, bands):
    mape_by_band = {}
    mdape_by_band = {}
    for band in bands.cat.categories:
        mask = (bands == band).values
        mape_by_band[band] = mean_absolute_percentage_error(y_true[mask], y_pred[mask])
        mdape_by_band[band] = np.median(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))
    return mape_by_band, mdape_by_band

model_predictions = {
    'Linear Regression': y_pred_lr,
    'Log-transformed Linear Regression': y_pred_log,
    'Decision Tree (depth 15)': y_pred_dt,
    'Random Forest (100 trees, depth 15)': y_pred_rf,
    'XGBoost (tuned)': y_pred_xgb,
    'LightGBM (tuned)': y_pred_lgbm,
}

mape_table = pd.DataFrame({name: band_metrics(y_test.values, pred, price_bands)[0] for name, pred in model_predictions.items()})
mdape_table = pd.DataFrame({name: band_metrics(y_test.values, pred, price_bands)[1] for name, pred in model_predictions.items()})

mape_table_fmt = mape_table.map(lambda x: f"{x:.2%}")
mdape_table_fmt = mdape_table.map(lambda x: f"{x:.2%}")

print("MAPE by price band:")
print(mape_table_fmt)
print()
print("MdAPE by price band:")
print(mdape_table_fmt)

MAPE by price band:
             Linear Regression Log-transformed Linear Regression  \
Q1 (lowest)             47.20%                            22.63%   
Q2                      32.09%                            17.50%   
Q3                      27.01%                            17.45%   
Q4                      21.92%                            17.07%   
Q5 (highest)            23.08%                            26.74%   

             Decision Tree (depth 15) Random Forest (100 trees, depth 15)  \
Q1 (lowest)                    21.29%                              20.37%   
Q2                             15.39%                              13.84%   
Q3                             15.17%                              12.72%   
Q4                             17.13%                              14.67%   
Q5 (highest)                   21.66%                              17.10%   

             XGBoost (tuned) LightGBM (tuned)  
Q1 (lowest)           14.81%           18.44%  
Q2          